# Private L4 deployment benchmark

Use only the rendered stage-specific handoff notebook in a fresh Colab **L4** runtime. Run all cells in order. This notebook verifies the reviewed runner and immutable parent A100 evidence, benchmarks the frozen calibration split, and creates a private verified package without training.

In [ ]:
import os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS'] = '1'
os.environ['MPLBACKEND'] = 'Agg'

from pathlib import Path
import hashlib, json, shutil, subprocess, sys

SOURCE_BUNDLE_SHA256 = "PASTE_FINAL_BUNDLE_SHA256"
RUNNER_GIT_SHA = "PASTE_FINAL_GIT_SHA"
PARENT_EXPERIMENT_GIT_SHA = "PASTE_PARENT_EXPERIMENT_GIT_SHA"
PARENT_DEPLOYMENT_GATE_SHA256 = "PASTE_PARENT_DEPLOYMENT_GATE_SHA256"
PARENT_CHECKPOINT_SHA256 = "PASTE_PARENT_CHECKPOINT_SHA256"
PARENT_ONNX_SHA256 = "PASTE_PARENT_ONNX_SHA256"
L4_HANDOFF_DIRECTORY = "PASTE_L4_HANDOFF_DIRECTORY"

immutable_values = (
    SOURCE_BUNDLE_SHA256,
    RUNNER_GIT_SHA,
    PARENT_EXPERIMENT_GIT_SHA,
    PARENT_DEPLOYMENT_GATE_SHA256,
    PARENT_CHECKPOINT_SHA256,
    PARENT_ONNX_SHA256,
    L4_HANDOFF_DIRECTORY,
)
if any(value.startswith('PASTE' + '_') for value in immutable_values):
    raise RuntimeError('Use the rendered L4 handoff notebook without manual editing')

DRIVE_ROOT = Path('/content/drive/MyDrive/pcb-defect-paired')
PARENT_DATASET = DRIVE_ROOT / 'dataset' / 'pcb'
HANDOFF_DIRECTORY = Path(L4_HANDOFF_DIRECTORY)
SOURCE_BUNDLE = HANDOFF_DIRECTORY / 'pcb-defect-source.bundle'
REPO = Path('/content/pcb-defect-l4-runner')
PARENT_WORKSPACE = (
    Path("/content/drive/MyDrive/pcb-defect-paired/workspaces")
    / PARENT_EXPERIMENT_GIT_SHA[:12]
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

from google.colab import drive
drive.mount('/content/drive')
if sha256_file(SOURCE_BUNDLE) != SOURCE_BUNDLE_SHA256:
    raise RuntimeError('Source bundle SHA-256 mismatch')


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', str(SOURCE_BUNDLE), str(REPO)], check=True)
    subprocess.run(['git', 'checkout', '--detach', RUNNER_GIT_SHA], cwd=REPO, check=True)
observed_runner = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()
if observed_runner != RUNNER_GIT_SHA:
    raise RuntimeError('Runner checkout has the wrong Git SHA; restart runtime')
status = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
if status:
    raise RuntimeError(f'Runner checkout is dirty: {status}')
symbolic_head = subprocess.run(['git', 'symbolic-ref', '-q', 'HEAD'], cwd=REPO, capture_output=True, text=True)
if symbolic_head.returncode == 0:
    raise RuntimeError('Runner checkout is not detached; restart runtime')

sys.path.insert(0, str(REPO / 'src'))
from pcb_defect.l4_contract import L4ParentIdentity, verify_l4_parent_inputs

parent = L4ParentIdentity.parse(
    experiment_git_sha=PARENT_EXPERIMENT_GIT_SHA,
    deployment_gate_sha256=PARENT_DEPLOYMENT_GATE_SHA256,
    checkpoint_sha256=PARENT_CHECKPOINT_SHA256,
    onnx_sha256=PARENT_ONNX_SHA256,
)
verify_l4_parent_inputs(PARENT_WORKSPACE, PARENT_DATASET, parent)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.11.18'], check=True)
UV = shutil.which('uv')
if UV is None:
    raise RuntimeError('uv is unavailable after the locked bootstrap')
subprocess.run([UV, 'sync', '--locked', '--no-editable', '--extra', 'train', '--group', 'eval', '--group', 'l4', '--reinstall-package', 'pcb-defect'], cwd=REPO, check=True)
VENV_PYTHON = REPO / '.venv' / 'bin' / 'python'
if not VENV_PYTHON.is_file():
    raise RuntimeError(f'Locked environment Python is missing: {VENV_PYTHON}')

def run_project_json(label: str, script: str, *arguments: object) -> dict[str, object]:
    command = [str(VENV_PYTHON), '-c', script, *(str(value) for value in arguments)]
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'{label} FAILED')
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'{label} returned no structured result')
    try:
        payload = json.loads(lines[-1])
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{label} returned invalid JSON') from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f'{label} returned a non-object result')
    return payload

TENSORRT_PROBE_SCRIPT = r'''
import json
import tensorrt as trt
if not trt.__version__ == '10.13.3.9':
    raise RuntimeError(f'unexpected TensorRT version: {trt.__version__}')
builder_available = bool(trt.Builder(trt.Logger()))
if not builder_available:
    raise RuntimeError('TensorRT Builder is unavailable')
print(json.dumps({'version': trt.__version__, 'builder_available': builder_available}, sort_keys=True))
'''
LOCKED_TENSORRT_STATE = run_project_json('LOCKED TENSORRT CONTRACT', TENSORRT_PROBE_SCRIPT)

VERIFY_INPUTS_SCRIPT = r'''
import json, sys
from pathlib import Path
from pcb_defect.l4_contract import L4RunIdentity, verify_l4_inputs
identity = L4RunIdentity.parse(
    runner_git_sha=sys.argv[4],
    experiment_git_sha=sys.argv[5],
    deployment_gate_sha256=sys.argv[6],
    checkpoint_sha256=sys.argv[7],
    onnx_sha256=sys.argv[8],
)
verified = verify_l4_inputs(
    Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), identity
)
print(json.dumps({
    'runner_git_sha': verified.runner_git_sha,
    'parent_experiment_git_sha': verified.parent.experiment_git_sha,
}, sort_keys=True))
'''
input_verification = run_project_json(
    'L4 input verification', VERIFY_INPUTS_SCRIPT, REPO, PARENT_WORKSPACE,
    PARENT_DATASET,
    RUNNER_GIT_SHA, PARENT_EXPERIMENT_GIT_SHA, PARENT_DEPLOYMENT_GATE_SHA256,
    PARENT_CHECKPOINT_SHA256, PARENT_ONNX_SHA256,
)
if input_verification != {
    'runner_git_sha': RUNNER_GIT_SHA,
    'parent_experiment_git_sha': PARENT_EXPERIMENT_GIT_SHA,
}:
    raise RuntimeError('Locked L4 input verification returned the wrong identity')

def run_streaming_command(
    command: list[str], *, cwd: Path, log_path: Path, label: str
) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as handle:
        handle.write(f'\n===== {label} attempt started =====\n')
        handle.flush()
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        if process.stdout is None:
            process.wait()
            raise RuntimeError(f'{label} did not expose a combined output stream')
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end='', flush=True)
        returncode = process.wait()
        handle.write(f'===== {label} attempt finished returncode={returncode} =====\n')
        handle.flush()
    if returncode:
        raise RuntimeError(f'{label} failed with returncode={returncode}; log: {log_path}')

def runtime_contract_state(label: str) -> dict[str, object]:
    command = [
        str(VENV_PYTHON),
        '-m',
        'pcb_defect.runtime_contract',
        '--require-cuda-provider',
    ]
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'{label} FAILED')
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'{label} returned no runtime state')
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{label} returned invalid runtime JSON') from exc

LOCKED_RUNTIME_STATE = runtime_contract_state('LOCKED RUNTIME CONTRACT')
print('RUNNER, PARENT, AND LOCKED ENVIRONMENT VERIFIED')


In [ ]:
benchmark_report_path = (
    PARENT_WORKSPACE / 'benchmark_l4' / RUNNER_GIT_SHA[:12] / 'benchmark_l4.json'
)
VERIFY_BENCHMARK_SCRIPT = r'''
import json, sys
from pathlib import Path
from pcb_defect.benchmark import benchmark_is_complete
from pcb_defect.l4_contract import L4RunIdentity
identity = L4RunIdentity.parse(
    runner_git_sha=sys.argv[5],
    experiment_git_sha=sys.argv[6],
    deployment_gate_sha256=sys.argv[7],
    checkpoint_sha256=sys.argv[8],
    onnx_sha256=sys.argv[9],
)
try:
    report = json.loads(Path(sys.argv[4]).read_text(encoding='utf-8'))
except (OSError, UnicodeError, json.JSONDecodeError):
    report = None
complete = benchmark_is_complete(
    Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), identity, report
)
print(json.dumps({'complete': complete}, sort_keys=True))
'''
benchmark_verification = run_project_json(
    'L4 benchmark verification', VERIFY_BENCHMARK_SCRIPT, REPO, PARENT_WORKSPACE,
    PARENT_DATASET, benchmark_report_path, RUNNER_GIT_SHA, PARENT_EXPERIMENT_GIT_SHA,
    PARENT_DEPLOYMENT_GATE_SHA256, PARENT_CHECKPOINT_SHA256, PARENT_ONNX_SHA256,
)
if benchmark_verification == {'complete': True}:
    print('SKIP completed verified L4 benchmark without mutating its packaged log')
else:
    benchmark_log = (
        PARENT_WORKSPACE / 'l4_logs' / RUNNER_GIT_SHA[:12] / 'benchmark_command.log'
    )
    BENCHMARK_RUNTIME_BEFORE = runtime_contract_state('BENCHMARK RUNTIME BEFORE')
    if BENCHMARK_RUNTIME_BEFORE != LOCKED_RUNTIME_STATE:
        raise RuntimeError('Locked runtime state changed before the L4 benchmark')
    benchmark_command = [
        str(VENV_PYTHON), '-m', 'pcb_defect.benchmark',
        '--repo', str(REPO),
        '--workspace', str(PARENT_WORKSPACE),
        '--dataset', str(PARENT_DATASET),
        '--expected-runner-git-sha', RUNNER_GIT_SHA,
        '--expected-experiment-git-sha', PARENT_EXPERIMENT_GIT_SHA,
        '--expected-deployment-gate-sha256', PARENT_DEPLOYMENT_GATE_SHA256,
        '--expected-checkpoint-sha256', PARENT_CHECKPOINT_SHA256,
        '--expected-onnx-sha256', PARENT_ONNX_SHA256,
        '--warmup', '30', '--cycles', '4',
    ]
    run_streaming_command(
        benchmark_command,
        cwd=REPO,
        log_path=benchmark_log,
        label='L4 benchmark',
    )
    BENCHMARK_RUNTIME_AFTER = runtime_contract_state('BENCHMARK RUNTIME AFTER')
    if BENCHMARK_RUNTIME_AFTER != BENCHMARK_RUNTIME_BEFORE:
        raise RuntimeError('Runtime state changed across the L4 benchmark command')
    benchmark_verification = run_project_json(
        'L4 benchmark verification', VERIFY_BENCHMARK_SCRIPT, REPO, PARENT_WORKSPACE,
        PARENT_DATASET, benchmark_report_path, RUNNER_GIT_SHA, PARENT_EXPERIMENT_GIT_SHA,
        PARENT_DEPLOYMENT_GATE_SHA256, PARENT_CHECKPOINT_SHA256, PARENT_ONNX_SHA256,
    )
    if benchmark_verification != {'complete': True}:
        raise RuntimeError('L4 benchmark did not produce complete verified evidence')

PACKAGE_ROOT = DRIVE_ROOT / 'packages'
package_command = [
    str(VENV_PYTHON), '-m', 'pcb_defect.l4_package',
    '--repo', str(REPO),
    '--workspace', str(PARENT_WORKSPACE),
    '--dataset', str(PARENT_DATASET),
    '--output-root', str(PACKAGE_ROOT),
    '--expected-runner-git-sha', RUNNER_GIT_SHA,
    '--expected-experiment-git-sha', PARENT_EXPERIMENT_GIT_SHA,
    '--expected-deployment-gate-sha256', PARENT_DEPLOYMENT_GATE_SHA256,
    '--expected-checkpoint-sha256', PARENT_CHECKPOINT_SHA256,
    '--expected-onnx-sha256', PARENT_ONNX_SHA256,
]
package_log = (
    PARENT_WORKSPACE / 'l4_logs' / RUNNER_GIT_SHA[:12] / 'package_command.log'
)
run_streaming_command(
    package_command,
    cwd=REPO,
    log_path=package_log,
    label='L4 package',
)
VERIFY_PACKAGE_SCRIPT = r'''
import hashlib, json, sys
from pathlib import Path
from pcb_defect.l4_contract import L4RunIdentity
from pcb_defect.l4_package import create_or_verify_l4_package
identity = L4RunIdentity.parse(
    runner_git_sha=sys.argv[5],
    experiment_git_sha=sys.argv[6],
    deployment_gate_sha256=sys.argv[7],
    checkpoint_sha256=sys.argv[8],
    onnx_sha256=sys.argv[9],
)
package = create_or_verify_l4_package(
    Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4]), identity
)
digest = hashlib.sha256(package.read_bytes()).hexdigest()
print(json.dumps({'package': str(package), 'sha256': digest}, sort_keys=True))
'''
package_verification = run_project_json(
    'L4 package verification', VERIFY_PACKAGE_SCRIPT, REPO, PARENT_WORKSPACE,
    PARENT_DATASET, PACKAGE_ROOT, RUNNER_GIT_SHA, PARENT_EXPERIMENT_GIT_SHA,
    PARENT_DEPLOYMENT_GATE_SHA256,
    PARENT_CHECKPOINT_SHA256, PARENT_ONNX_SHA256,
)
if set(package_verification) != {'package', 'sha256'}:
    raise RuntimeError('L4 package verification returned malformed structured data')
package = Path(str(package_verification['package']))
package_sha256 = package_verification['sha256']
if not isinstance(package_sha256, str):
    raise RuntimeError('L4 package verification returned an invalid SHA-256')
print('L4 HANDOFF COMPLETE', package, package_sha256)
